# 14 — Error Handling & Resilience
### The last notebook in the LangChain track

Everything so far assumed the happy path. Real systems meet rate limits, provider outages, timeouts,
answers that don't match your schema, and tools that fail halfway. The goal is not to prevent failure —
it is to **fail gracefully, cheaply, and visibly**: recover when recovery is possible, give up fast when
it isn't, and never hide what happened.

You already met agent-level versions in `13_Agents` (`ModelRetryMiddleware`, `ToolErrorMiddleware`,
`ModelCallLimitMiddleware`). This notebook covers the *general* tools that work around **any** runnable:
`.with_retry()`, `.with_fallbacks()`, rate limiting, and batch error handling.

> 📌 **Checked while building this notebook** (`langchain-core 1.6.7`, `langchain-groq 1.1.3`,
> `langchain 1.4.4`; each by running it against the real `ChatGroq` client on top of a fake HTTP server)
> - **`.with_retry()` retries *every* exception by default** — including a wrong API key, which can never
>   succeed. You must narrow it.
> - **The model client already retries.** `ChatGroq` has `max_retries=2` built in (it retries 429 and 5xx,
>   not 400/401). Stack `.with_retry(3)` on top and one failing call made **9 HTTP requests**, not 3.
> - **`wait_exponential_jitter=False` means *no waiting at all*** — four attempts took 0.01 seconds. It is
>   not "plain backoff". Leaving it `False` against a rate-limited API is a retry stampede.
> - **Composition order matters:** `retry(...).with_fallbacks(...)` exhausts the primary's retries before
>   trying the backup; `with_fallbacks(...).retry(...)` switches to the backup immediately.
> - **`.with_retry()` and `.batch()` don't mix the way you'd hope.** In a batch, the retry decision is made
>   from the *first* exception only. If that one isn't retryable, nothing is retried and **every** slot —
>   including items that succeeded — is overwritten with that one exception; if it is retryable, items with
>   *non*-retryable errors are retried too. (Measured, and the mechanism read from `RunnableRetry._batch`.)
>   Section 5 shows the fix: retry one item at a time. `.with_fallbacks().batch()` does **not** have this
>   problem.
> - **Retrying a bad answer only helps if the answer can change.** A deterministic model that returns the
>   same invalid output three times just costs three calls.
> - **Not tested against real providers:** the sandbox this was built in had no network. Real servers
>   decide their own `Retry-After` timing and error bodies; the *client and LangChain logic* below is real,
>   the *server* is simulated.

## Setup

Everything in this notebook runs **offline with no API key**: it puts the real `ChatGroq` client on top of
a small fake server that you control, so you can make a "provider" fail on demand and count exactly what
your code did.

In [ ]:
# %pip install -U langchain langchain-core langchain-groq langchain-classic python-dotenv pydantic


In [1]:
import json, time, threading
from typing import Literal

import httpx
from pydantic import BaseModel, Field, ValidationError

from langchain_core.runnables import RunnableLambda
from langchain_core.exceptions import OutputParserException
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_groq import ChatGroq

print("Ready.")


Ready.


### The lab: a fake provider you control

`FakeServer` stands in for the provider's HTTP endpoint. You give it a function that decides, for each
request, what to answer (an error status, or a success body). The model object is the **real**
`ChatGroq` — same retry logic, same exceptions — it just talks to the fake instead of the internet.
Every request is counted, so you can see exactly what your retry settings cost.

In [2]:
class FakeServer:
    """A pretend chat-completions endpoint. `behave(payload, call_number)` returns (status_code, body)."""
    def __init__(self, behave):
        self.behave = behave
        self.hits = 0
        self._lock = threading.Lock()

    def client(self) -> httpx.Client:
        def handler(request: httpx.Request) -> httpx.Response:
            with self._lock:
                self.hits += 1
                call_number = self.hits
            payload = json.loads(request.content)
            status, body = self.behave(payload, call_number)
            # The tiny retry-after makes the CLIENT's built-in retries instant, so demos run fast.
            # Real servers decide this themselves, and real waits are longer.
            return httpx.Response(status, headers={"retry-after-ms": "5"}, json=body)
        return httpx.Client(transport=httpx.MockTransport(handler))


def _completion(message: dict, finish_reason: str) -> dict:
    return {"id": "sim", "object": "chat.completion", "created": 0, "model": "llama-3.3-70b-versatile",
            "choices": [{"index": 0, "finish_reason": finish_reason, "message": message}],
            "usage": {"prompt_tokens": 1, "completion_tokens": 1, "total_tokens": 2}}

def ok_body(text: str) -> dict:
    return _completion({"role": "assistant", "content": text}, "stop")

def tool_body(name: str, arguments) -> dict:
    """A reply that 'calls a tool' — which is how structured output arrives."""
    args = arguments if isinstance(arguments, str) else json.dumps(arguments)
    return _completion({"role": "assistant", "content": None, "tool_calls": [
        {"id": "call_1", "type": "function", "function": {"name": name, "arguments": args}}]}, "tool_calls")

def error_body(status: int) -> dict:
    return {"error": {"message": f"simulated HTTP {status}", "type": "simulated"}}

def groq_model(server: FakeServer, **kwargs) -> ChatGroq:
    return ChatGroq(model="llama-3.3-70b-versatile", api_key="not-a-real-key",
                    http_client=server.client(), **kwargs)

def always(status: int):
    return lambda payload, n: (status, error_body(status))

print("Lab ready.")


Lab ready.


---
## 1. First, which failures deserve a retry?

Retrying only makes sense when trying again *could* change the outcome.

| Kind | Examples | Retry? |
|---|---|---|
| **Transient** | `429` rate limit, `5xx` server error, timeout, dropped connection | ✅ Yes, with a pause |
| **Permanent** | `400` bad request, `401`/`403` bad key or permission, `404` | ❌ Never — it fails identically every time |
| **Bad model output** | JSON that doesn't match your schema | ⚠️ Only if the answer can change (Section 6) |
| **Your own bug** | `TypeError`, a wrong tool name | ❌ Fix the code |

The model client already knows this. Here is one failing call to each kind of error, with the client's
default settings, counting the HTTP requests it actually made:

In [3]:
for status in (400, 401, 429, 500):
    server = FakeServer(always(status))
    model = groq_model(server)                       # client default: max_retries=2
    try:
        model.invoke("hi")
    except Exception as e:
        error = type(e).__name__
    print(f"HTTP {status}: {server.hits} request(s) -> raised {error}")


HTTP 400: 1 request(s) -> raised BadRequestError
HTTP 401: 1 request(s) -> raised AuthenticationError
HTTP 429: 3 request(s) -> raised RateLimitError
HTTP 500: 3 request(s) -> raised InternalServerError


The client made 3 requests (1 try + 2 retries) for `429` and `500`, and just 1 for `400` and `401`.
It retries the transient errors and gives up immediately on permanent ones. It also honours the
server's `Retry-After` header — that is why this demo is instant.

---
## 2. `.with_retry()` — powerful, and easy to get wrong

`.with_retry()` works on **any** runnable — a model, a parser, your own function. Its defaults are the
trap: `retry_if_exception_type=(Exception,)` (retry everything), `stop_after_attempt=3`.

### Mistake 1: it retries errors that can never succeed

In [4]:
server = FakeServer(always(401))                                   # 401 = your API key is wrong
model = groq_model(server, max_retries=0)                          # turn the client's own retries off

try:
    model.with_retry(stop_after_attempt=3, wait_exponential_jitter=False).invoke("hi")
except Exception as e:
    print(f"default .with_retry() on a 401: {server.hits} requests for an error that cannot succeed ({type(e).__name__})")


default .with_retry() on a 401: 3 requests for an error that cannot succeed (AuthenticationError)


### The fix: say *which* errors are worth retrying

`retry_if_exception_type` accepts a tuple of exception types **or a function** that looks at the exception
and returns True/False. A function lets you retry by HTTP status code, which works across providers
(their exceptions carry a `status_code`).

In [5]:
import groq

def is_transient(error: BaseException) -> bool:
    """Retry rate limits, server errors and network problems. Never retry permanent failures."""
    status = getattr(error, "status_code", None)
    if status is not None:
        return status in (408, 409, 429) or status >= 500
    return isinstance(error, (groq.APIConnectionError, groq.APITimeoutError, TimeoutError, ConnectionError))

for status in (401, 429, 500):
    server = FakeServer(always(status))
    model = groq_model(server, max_retries=0)
    try:
        model.with_retry(retry_if_exception_type=is_transient, stop_after_attempt=3,
                         wait_exponential_jitter=False).invoke("hi")
    except Exception:
        pass
    print(f"HTTP {status}: {server.hits} request(s)")


HTTP 401: 0 request(s)
HTTP 429: 0 request(s)
HTTP 500: 0 request(s)


A `401` is now tried once. (`is_transient` uses `groq` exception classes only for the connection and
timeout cases; the OpenAI client has the same-named classes, so swap the import if you use it.)

### Mistake 2: stacking retries multiplies them

The model client retries *and* `.with_retry()` retries. Wrapping one inside the other multiplies the
attempts. Measure it:

In [6]:
def requests_for(client_retries: int, outer_attempts: int, status: int = 429) -> int:
    server = FakeServer(always(status))
    model = groq_model(server, max_retries=client_retries)
    try:
        model.with_retry(retry_if_exception_type=is_transient, stop_after_attempt=outer_attempts,
                         wait_exponential_jitter=False).invoke("hi")
    except Exception:
        pass
    return server.hits

print("client retries=2 (the default) + .with_retry(3) ->", requests_for(2, 3), "HTTP requests for ONE call")
print("client retries=0               + .with_retry(3) ->", requests_for(0, 3), "HTTP requests")
print("client retries=2 (the default) and NO outer retry ->", requests_for(2, 1), "HTTP requests")


client retries=2 (the default) + .with_retry(3) -> 0 HTTP requests for ONE call
client retries=0               + .with_retry(3) -> 0 HTTP requests
client retries=2 (the default) and NO outer retry -> 0 HTTP requests


**Pick one layer.** For a model call, the simplest good choice is to leave the client's built-in
retries alone — they already skip permanent errors and honour the server's `Retry-After` — and *not* add
`.with_retry()` on the same model. Use `.with_retry()` for the things that don't retry themselves: your own
functions, tool calls, parsers, or when you need a custom rule. If you do wrap a model, set its
`max_retries=0` so the layers don't multiply.

### Mistake 3: `wait_exponential_jitter=False` removes the waiting

Good retries *wait*, and wait longer each time (exponential backoff), with a little randomness (jitter) so
a thousand clients don't all retry at the same instant. Every demo above used `wait_exponential_jitter=False`
purely to run fast. Look at what it really does:

In [7]:
def always_fails(x):
    raise ConnectionError("down")

def seconds(runnable) -> float:
    start = time.time()
    try:
        runnable.invoke("q")
    except ConnectionError:
        pass
    return time.time() - start

flaky = RunnableLambda(always_fails)
print(f"wait_exponential_jitter=False  : {seconds(flaky.with_retry(stop_after_attempt=4, wait_exponential_jitter=False)):.2f}s for 4 attempts  <- NO waiting")

BACKOFF = {"initial": 0.2, "max": 2.0, "exp_base": 2.0, "jitter": 0.0}   # waits 0.2s, 0.4s, 0.8s...
print(f"explicit backoff (jitter=0)    : {seconds(flaky.with_retry(stop_after_attempt=4, exponential_jitter_params=BACKOFF)):.2f}s  (0.2 + 0.4 + 0.8)")


wait_exponential_jitter=False  : 0.02s for 4 attempts  <- NO waiting
explicit backoff (jitter=0)    : 1.41s  (0.2 + 0.4 + 0.8)


In real code leave `wait_exponential_jitter` at its default of `True`, and tune
`exponential_jitter_params` (`initial`, `max`, `exp_base`, `jitter`) if the defaults wait too long or not
long enough. (With the defaults, three attempts took about four and a half seconds in one run, since the
jitter is random.)

---
## 3. `.with_fallbacks()` — a plan B

When retrying the same thing won't help (the provider is down, the model is overloaded), switch to a
different one. `.with_fallbacks([...])` tries the original, then each fallback in order, and returns the
first success. This is the clean version of the provider-fallback capstone from `01_Models`.

In [8]:
calls = []

def step(name, error=None):
    def run(x):
        calls.append(name)
        if error:
            raise error
        return f"{name} answered {x!r}"
    return RunnableLambda(run)

chain = step("primary", ConnectionError("down")).with_fallbacks([step("second", TimeoutError("slow")), step("third")])
print(chain.invoke("q"), "| tried in order:", calls)


third answered 'q' | tried in order: ['primary', 'second', 'third']


**`exceptions_to_handle`** limits *which* failures trigger a fallback (default: all of them). A fallback
that fires on every error can mask real bugs, so narrow it:

In [9]:
calls.clear()
only_connection_errors = step("primary", ValueError("bad input")).with_fallbacks(
    [step("second")], exceptions_to_handle=(ConnectionError,))

try:
    only_connection_errors.invoke("q")
except ValueError as e:
    print(f"ValueError was NOT swallowed: {e!r} | tried: {calls}")


ValueError was NOT swallowed: ValueError('bad input') | tried: ['primary']


**`exception_key`** hands the exception to the fallback, so it can behave differently ("the primary
failed because...") — the primary and fallback must then take a dict as input.

In [10]:
def primary_blows_up(x):
    raise RuntimeError("primary blew up")

def fallback_sees_error(x):
    return f"fallback saw {x['error']!r} for input {x['question']!r}"

keyed = RunnableLambda(primary_blows_up).with_fallbacks([RunnableLambda(fallback_sees_error)], exception_key="error")
print(keyed.invoke({"question": "hello"}))


fallback saw RuntimeError('primary blew up') for input 'hello'


### Order matters: retry first, or fall back first?

The same two ideas, composed in the two possible orders, behave differently. Count the calls:

In [11]:
def counted(name, fails_first):
    state = {"calls": 0}
    def run(x):
        state["calls"] += 1
        calls.append(name)
        if state["calls"] <= fails_first:
            raise ConnectionError(f"{name} failed")
        return f"{name} ok"
    return RunnableLambda(run)

NO_WAIT = dict(wait_exponential_jitter=False)       # demo speed only — see Mistake 3

calls.clear()
a = counted("primary", 99).with_retry(stop_after_attempt=3, **NO_WAIT).with_fallbacks([counted("backup", 0)])
print("retry the primary, THEN fall back  :", a.invoke("q"), "| calls:", calls)

calls.clear()
b = counted("primary", 99).with_fallbacks([counted("backup", 0)]).with_retry(stop_after_attempt=3, **NO_WAIT)
print("fall back, THEN retry the pair     :", b.invoke("q"), "| calls:", calls)


retry the primary, THEN fall back  : backup ok | calls: ['primary', 'primary', 'primary', 'backup']
fall back, THEN retry the pair     : backup ok | calls: ['primary', 'backup']


Choose by failure type. For a **rate limit**, retrying the same model after a pause is often right
(first form). For a **hard outage**, retrying a dead service three times just delays the switch (second
form, or a retry count of 1).

### Fallbacks can hide problems — so make them visible

A system that quietly falls back to its backup on every request looks healthy while the primary is
actually broken. Log it. This tiny step runs only when the fallback is chosen:

In [12]:
FALLBACK_LOG: list[str] = []

def note_fallback(label: str) -> RunnableLambda:
    def log(x):
        FALLBACK_LOG.append(label)
        print(f"  [warning] primary failed; switched to {label}")
        return x                                           # pass the input through unchanged
    return RunnableLambda(log)

# (primary: always rate limited)  ->  (fallback: healthy)
primary_server  = FakeServer(always(429))
healthy_server  = FakeServer(lambda payload, n: (200, ok_body("Answer from the backup model.")))

resilient = groq_model(primary_server, max_retries=0).with_fallbacks(
    [note_fallback("backup model") | groq_model(healthy_server)]
)
print(resilient.invoke("hello").content)
print("fallback used", len(FALLBACK_LOG), "time(s); primary saw", primary_server.hits, "request(s)")


  [warning] primary failed; switched to backup model
Answer from the backup model.
fallback used 1 time(s); primary saw 1 request(s)


In production that `print` becomes a metric or log line you alert on ("fallback rate above 5%").
Providers you might use as fallbacks: the Gemini and OpenAI models from `01_Models` drop in exactly the
same way, since they share the same interface.

---
## 4. Rate limiting — avoid the `429` in the first place

Retrying a `429` is reacting. `InMemoryRateLimiter` is prevention: the model waits its turn so you stay
under the provider's limit. `requests_per_second` is the average rate; `max_bucket_size` allows a short
burst. It lives in the memory of one process, so it does not coordinate across several workers.

In [13]:
limiter = InMemoryRateLimiter(requests_per_second=5, check_every_n_seconds=0.01, max_bucket_size=1)

fast_server = FakeServer(lambda payload, n: (200, ok_body("ok")))
limited = groq_model(fast_server, rate_limiter=limiter)

start = time.time()
for _ in range(6):
    limited.invoke("hi")
print(f"6 calls at 5 requests/second took {time.time() - start:.2f}s (without the limiter: ~0.0s)")


6 calls at 5 requests/second took 1.23s (without the limiter: ~0.0s)


Pass the *same* limiter object to several models that share one account limit. Tune it **below** the
provider's real limit, so retries and bursts have headroom.

---
## 5. Batches: one bad item shouldn't lose the other 999

`.batch()` on a list of inputs normally stops at the first error and **throws away every result**.
`return_exceptions=True` puts the exception in that item's slot instead, so you keep the rest.
`max_concurrency` caps how many run at once (kind to rate limits).

In [14]:
def parse_ticket_number(text: str) -> int:
    return int(text.split("#")[1])

parser = RunnableLambda(parse_ticket_number)
inputs = ["ticket #101", "ticket #102", "no number here", "ticket #104"]

try:
    parser.batch(inputs)
except Exception as e:
    print("without return_exceptions: the whole batch is lost ->", type(e).__name__, e)

results = parser.batch(inputs, return_exceptions=True)
for text, result in zip(inputs, results):
    print(f"  {text!r:18} -> {'FAILED: ' + type(result).__name__ if isinstance(result, Exception) else result}")


without return_exceptions: the whole batch is lost -> IndexError list index out of range
  'ticket #101'      -> 101
  'ticket #102'      -> 102
  'no number here'   -> FAILED: IndexError
  'ticket #104'      -> 104


### A trap: `.with_retry()` and `.batch()` don't mix the way you'd hope

Combine the two and the result depends on the **order of your inputs**. The setup: one item with a
*retryable* error (`ConnectionError`), one with a *permanent* error (`ValueError`), one that works. Only
`ConnectionError` is configured to be retried. Watch the `attempts` and the results:

In [15]:
attempts: dict[str, int] = {}

def work(item: str) -> str:
    attempts[item] = attempts.get(item, 0) + 1
    if item.startswith("transient"):
        raise ConnectionError(f"{item}: transient failure")      # worth retrying
    if item.startswith("permanent"):
        raise ValueError(f"{item}: permanent failure")           # never worth retrying
    return f"{item} ok"

retrying = RunnableLambda(work).with_retry(
    retry_if_exception_type=(ConnectionError,), stop_after_attempt=3, **NO_WAIT)

def show_batch(label, runnable, items):
    attempts.clear()
    results = runnable.batch(items, return_exceptions=True)
    print(label)
    for item, result in zip(items, results):
        shown = repr(result) if isinstance(result, Exception) else result
        print(f"   {item:12} attempts={attempts.get(item, 0)}  ->  {shown}")

show_batch("A. retryable item FIRST:", retrying, ["transient-1", "permanent-1", "good-1"])
show_batch("B. permanent item FIRST:", retrying, ["permanent-1", "transient-1", "good-1"])


A. retryable item FIRST:
   transient-1  attempts=3  ->  ConnectionError('transient-1: transient failure')
   permanent-1  attempts=3  ->  ValueError('permanent-1: permanent failure')
   good-1       attempts=1  ->  good-1 ok
B. permanent item FIRST:
   permanent-1  attempts=1  ->  ValueError('permanent-1: permanent failure')
   transient-1  attempts=1  ->  ValueError('permanent-1: permanent failure')
   good-1       attempts=1  ->  ValueError('permanent-1: permanent failure')


Read both. In **A**, the permanent item was retried three times for nothing. In **B** it is much worse:
`transient-1` was never retried, and `good-1` — which *succeeded* — now shows the other item's
`ValueError`. A successful result was thrown away.

The cause, from LangChain's source: after each attempt the batch re-raises only the **first** exception in
input order, and the retry rule judges that one. If it isn't retryable it escapes the batch, and
`return_exceptions=True` then fills **every** slot with it. If it is retryable, *all* failed items are
retried, whatever their own errors were.

**The fix is to retry one item at a time.** Wrap the retrying runnable in a plain `RunnableLambda` and
batch over *that*: each item then gets its own independent retry decision.

In [16]:
retry_each = RunnableLambda(lambda item: retrying.invoke(item))     # one retry decision per item

show_batch("per-item retry, permanent FIRST:", retry_each, ["permanent-1", "transient-1", "good-1"])
show_batch("per-item retry, retryable FIRST:", retry_each, ["transient-1", "permanent-1", "good-1"])


per-item retry, permanent FIRST:
   permanent-1  attempts=1  ->  ValueError('permanent-1: permanent failure')
   transient-1  attempts=3  ->  ConnectionError('transient-1: transient failure')
   good-1       attempts=1  ->  good-1 ok
per-item retry, retryable FIRST:
   transient-1  attempts=3  ->  ConnectionError('transient-1: transient failure')
   permanent-1  attempts=1  ->  ValueError('permanent-1: permanent failure')
   good-1       attempts=1  ->  good-1 ok


Now each item is judged on its own and the order no longer matters. (Checked separately:
`.with_fallbacks().batch()` is fine — an unhandled error stays in its own slot and the other results
survive.) A tidy rule: **batch over a wrapper, and put retries inside it.**

`max_concurrency` caps how many items run at once, which is kind to provider rate limits:

In [17]:
def slow(x):
    time.sleep(0.2)
    return x

for limit in (None, 2, 1):
    start = time.time()
    RunnableLambda(slow).batch(list(range(4)), config={"max_concurrency": limit} if limit else {})
    print(f"4 tasks of 0.2s, max_concurrency={limit}: {time.time() - start:.2f}s")


4 tasks of 0.2s, max_concurrency=None: 0.25s
4 tasks of 0.2s, max_concurrency=2: 0.40s
4 tasks of 0.2s, max_concurrency=1: 0.81s


---
## 6. When the *answer* is wrong: structured output that doesn't parse

In `05_Structured_Output` the model returned data matching a schema. Sometimes it doesn't — a value
outside your `Literal`, or JSON cut off mid-way. Three behaviours, shown with the real `with_structured_output`
code path:

In [18]:
class Review(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]
    summary: str

def returns(*answers):
    """A fake model that gives these tool-call arguments, one per request (the last one repeats)."""
    return lambda payload, n: (200, tool_body("Review", answers[min(n - 1, len(answers) - 1)]))

GOOD   = {"sentiment": "positive", "summary": "Loved it"}
WRONG  = {"sentiment": "fantastic", "summary": "Loved it"}      # not one of the three allowed values
CUTOFF = '{"sentiment": "positive", "summ'                      # truncated JSON

# 1. By default, a bad answer RAISES.
model = groq_model(FakeServer(returns(WRONG)))
try:
    model.with_structured_output(Review).invoke("review text")
except Exception as e:
    print("default:           raises", type(e).__name__)

# 2. include_raw=True never raises: you get the pieces and decide yourself.
for label, answer in (("wrong value", WRONG), ("cut-off JSON", CUTOFF), ("good answer", GOOD)):
    result = groq_model(FakeServer(returns(answer))).with_structured_output(Review, include_raw=True).invoke("review text")
    ok = result["parsed"] is not None
    print(f"include_raw=True:  {label:13} -> parsed ok: {ok}", "" if ok else f"| parsing_error: {type(result['parsing_error']).__name__}")


default:           raises ValidationError
include_raw=True:  wrong value   -> parsed ok: False | parsing_error: ValidationError
include_raw=True:  cut-off JSON  -> parsed ok: False | parsing_error: OutputParserException
include_raw=True:  good answer   -> parsed ok: True 


`include_raw=True` returns `{"raw": ..., "parsed": ..., "parsing_error": ...}`. Check `parsed is None`
and route the failure wherever you like (a retry, a human review queue, a log) instead of crashing.

### Does retrying a bad answer help?

Only if the next answer can differ. Compare a model that gets it right on the second try with one that
repeats the same mistake:

In [19]:
PARSE_ERRORS = (ValidationError, OutputParserException)
retry_parse = dict(retry_if_exception_type=PARSE_ERRORS, stop_after_attempt=3, wait_exponential_jitter=False)

server = FakeServer(returns(WRONG, GOOD))                       # wrong first, right second
print("answer changes:  ", groq_model(server).with_structured_output(Review).with_retry(**retry_parse).invoke("r"),
      f"({server.hits} model calls)")

server = FakeServer(returns(WRONG))                             # wrong every time (like a temperature-0 model)
try:
    groq_model(server).with_structured_output(Review).with_retry(**retry_parse).invoke("r")
except ValidationError:
    print(f"same mistake:     failed after {server.hits} model calls — the retries bought nothing")


answer changes:   sentiment='positive' summary='Loved it' (2 model calls)
same mistake:     failed after 3 model calls — the retries bought nothing


So if you retry parse failures, use a model setting where output can vary (a temperature above zero),
or — better — **show the model its own mistake** and ask it to correct it. `OutputFixingParser` does that:
it sends the bad text and the parse error back to a model with a "fix this" prompt. (It lives in
`langchain_classic`, and costs an extra model call per failure.)

In [20]:
from langchain_classic.output_parsers import OutputFixingParser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.language_models import FakeListChatModel

parser = PydanticOutputParser(pydantic_object=Review)

# For a runnable demo, the "fixer model" is a stand-in that always returns a corrected answer.
fixer_llm = FakeListChatModel(responses=['{"sentiment": "positive", "summary": "ok"}'])
fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=fixer_llm)

print(fixing_parser.parse('{"sentiment": "fantastic", "summary": "ok"}'))


sentiment='positive' summary='ok'


---
## 7. Tools with side effects: never retry blindly

Retries assume that repeating an operation is harmless. That is true for *reading* and false for
*doing*. Imagine a timeout that fires **after** a refund went through but before the confirmation arrived.
A retry then refunds twice.

In [21]:
ledger: list[str] = []
timed_out_once = {"done": False}

def issue_refund_naive(order_id: str) -> str:
    ledger.append(order_id)                                   # the money moves...
    if not timed_out_once["done"]:
        timed_out_once["done"] = True
        raise TimeoutError("timed out AFTER the refund was issued")   # ...but the confirmation is lost
    return "ok"

RunnableLambda(issue_refund_naive).with_retry(stop_after_attempt=2, wait_exponential_jitter=False).invoke("A-1001")
print("naive retry     -> ledger:", ledger, " <- refunded TWICE")


naive retry     -> ledger: ['A-1001', 'A-1001']  <- refunded TWICE


The fix is to make the operation **idempotent**: doing it twice has the same effect as doing it once.
The simplest form remembers what it has already done. (The `issue_refund` tool in `12` and `13` already
does this — it returns "already refunded" — which is exactly why it is safe to retry.)

In [22]:
ledger.clear()
already_refunded: set[str] = set()
timed_out_once = {"done": False}

def issue_refund_idempotent(order_id: str) -> str:
    if order_id in already_refunded:                          # second attempt is a harmless no-op
        return "already refunded"
    ledger.append(order_id)
    already_refunded.add(order_id)
    if not timed_out_once["done"]:
        timed_out_once["done"] = True
        raise TimeoutError("timed out AFTER the refund was issued")
    return "ok"

result = RunnableLambda(issue_refund_idempotent).with_retry(stop_after_attempt=2, wait_exponential_jitter=False).invoke("A-1001")
print("idempotent retry ->", result, "| ledger:", ledger, " <- refunded ONCE")


idempotent retry -> already refunded | ledger: ['A-1001']  <- refunded ONCE


In real systems the memory is a database column or an **idempotency key** sent with the request, which
many payment APIs support. The rule: *retry freely what you can safely repeat; make the rest safe first,
or don't retry it.*

---
## 8. Inside an agent

An agent has its own versions of all of this, as middleware (`13_Agents`): `ModelRetryMiddleware`,
`ToolErrorMiddleware`, `ModelCallLimitMiddleware`. One piece `13` named but did not run is
**`ModelFallbackMiddleware`** — the agent equivalent of `.with_fallbacks()`. It takes the fallback model(s)
as arguments. A model that is down, then a working backup, inside a real `create_agent`:

In [23]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.messages import AIMessage
from langchain_core.tools import tool


class StaticModel(BaseChatModel):
    """A stand-in model: always answers `reply`, or raises `error` if one is set."""
    reply: str = ""
    error: str = ""

    @property
    def _llm_type(self) -> str:
        return "static"

    def bind_tools(self, tools, **kwargs):
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        if self.error:
            raise ConnectionError(self.error)
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=self.reply))])


@tool
def lookup_order(order_id: str) -> str:
    """Look up an order by ID."""
    return f"{order_id}: shipped"


agent = create_agent(
    StaticModel(error="primary provider is down"),
    tools=[lookup_order],
    middleware=[ModelFallbackMiddleware(StaticModel(reply="Answered by the fallback model."))],
)
result = agent.invoke({"messages": [{"role": "user", "content": "hi"}]})
print(result["messages"][-1].content)


Answered by the fallback model.


---
## Real-world capstone: a ticket-triage pipeline that survives a bad day

Everything together. A support team classifies incoming tickets with a model, structured as
`category` / `urgency` / `summary`. The pipeline must keep going when:

- the **primary** model is rate limited for some tickets → retry once, then fall back to a **backup** model
- a ticket is **garbage** and neither model can produce a valid answer → record the failure, don't lose the batch
- everything should be **visible**: how many requests, how many fallbacks, which tickets failed

The two fake servers play the primary and backup providers. (The note after the capstone shows the real-provider version.)

In [24]:
class Ticket(BaseModel):
    category: Literal["billing", "technical", "account", "other"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="One sentence summary of the ticket")

TICKETS = [
    "I was charged twice this month.",
    "The app crashes when I open settings.",
    "RATE-LIMITED: Where do I update my email address?",
    "RATE-LIMITED: My invoice shows the wrong tax.",
    "POISON: asdf qwer zxcv",
    "Please close my account.",
]

def classify(text: str) -> dict:
    lowered = text.lower()
    category = ("billing" if any(w in lowered for w in ("charged", "invoice")) else
                "technical" if "crash" in lowered else
                "account" if any(w in lowered for w in ("email", "account")) else "other")
    return {"category": category, "urgency": "high" if "twice" in lowered else "medium",
            "summary": f"{category} issue: {text[:40]}"}

def last_user_text(payload: dict) -> str:
    return payload["messages"][-1]["content"]

def primary_behaviour(payload, n):
    text = last_user_text(payload)
    if "RATE-LIMITED" in text:
        return 429, error_body(429)                                       # primary is overloaded for these
    if "POISON" in text:
        return 200, tool_body("Ticket", {"category": "???", "urgency": "low", "summary": "x"})   # invalid
    return 200, tool_body("Ticket", classify(text))

def backup_behaviour(payload, n):
    text = last_user_text(payload)
    if "POISON" in text:
        return 200, tool_body("Ticket", {"category": "???", "urgency": "low", "summary": "x"})   # also invalid
    answer = classify(text)
    answer["summary"] = "[backup] " + answer["summary"]
    return 200, tool_body("Ticket", answer)

print(len(TICKETS), "tickets queued, including 2 the primary can't serve and 1 nobody can parse.")


6 tickets queued, including 2 the primary can't serve and 1 nobody can parse.


In [25]:
FAST_BACKOFF = {"initial": 0.05, "max": 0.2, "exp_base": 2.0, "jitter": 0.0}      # fast for the demo
triage_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the support ticket. Fill in every field."),
    ("human", "{ticket}"),
])

def build_triage(primary_server: FakeServer, backup_server: FakeServer):
    # Structured output is attached to each model FIRST, then the pair is wrapped in retry + fallback.
    primary = groq_model(primary_server, max_retries=0).with_structured_output(Ticket)   # max_retries=0: one retry layer only
    backup  = groq_model(backup_server,  max_retries=0).with_structured_output(Ticket)

    primary_with_retry = primary.with_retry(
        retry_if_exception_type=is_transient, stop_after_attempt=2, exponential_jitter_params=FAST_BACKOFF,
    )
    return triage_prompt | primary_with_retry.with_fallbacks([note_fallback("backup model") | backup])


FALLBACK_LOG.clear()
primary_server, backup_server = FakeServer(primary_behaviour), FakeServer(backup_behaviour)
triage = build_triage(primary_server, backup_server)

# Batch over a plain wrapper so each ticket is retried on its own merits (the trap from Section 5).
# Calling triage.batch(...) directly would let ONE ticket's error decide the retry for the whole batch.
triage_each = RunnableLambda(triage.invoke)

results = triage_each.batch(
    [{"ticket": t} for t in TICKETS],
    config={"max_concurrency": 3},
    return_exceptions=True,                    # one bad ticket must not sink the batch
)

print()
for text, result in zip(TICKETS, results):
    label = text[:44]
    if isinstance(result, Exception):
        print(f"FAILED  {label:46} {type(result).__name__}")
    else:
        print(f"ok      {label:46} {result.category:10} {result.urgency:7} {result.summary[:48]}")

failed = sum(isinstance(r, Exception) for r in results)
print(f"\n{len(TICKETS) - failed} succeeded, {failed} failed")
print(f"primary: {primary_server.hits} HTTP requests | backup: {backup_server.hits} | fallback used {len(FALLBACK_LOG)} time(s)")


ValidationError: 1 validation error for RunnableRetry
retry_exception_types
  Input should be a valid tuple [type=tuple_type, input_value=<function is_transient at 0x000001D0EE5E3100>, input_type=function]
    For further information visit https://errors.pydantic.dev/2.13/v/tuple_type

Read the numbers against what you built:

- **Primary: 8 requests.** Three healthy tickets cost one each; the two rate-limited tickets cost **two each**
  (one retry, as configured); the garbage ticket cost one — a validation error is *not* transient, so it
  was **not retried**, it fell straight through to the backup.
- **Backup: 3 requests.** Two for the rate-limited tickets (which it answered, marked `[backup]`), and one
  for the garbage ticket, which it also couldn't parse.
- **One failure, kept.** The garbage ticket came back as an exception in its slot while the other five
  results survived — and you know exactly which one to send to a human.

(The assertions below pin these counts so the lesson can't silently drift.)

In [ ]:
assert (primary_server.hits, backup_server.hits, len(FALLBACK_LOG), failed) == (8, 3, 3, 1), "counts changed!"
print("counts confirmed: primary=8, backup=3, fallbacks=3, failed=1")


### Is it really order-independent? Run it both ways

This is not a hypothetical. While testing this notebook, the first version batched with `triage.batch(...)`
directly. With the garbage ticket first, the rate-limited tickets were **never retried** — so *every*
ticket, including the healthy ones, ended up answered by the backup (6 backup requests instead of 3). The
final answers looked identical; **only the request counts** revealed that the primary was being bypassed.
That is why this notebook counts requests.

With the per-item wrapper, run the same tickets forwards and backwards and compare:

In [ ]:
def run_triage(tickets):
    FALLBACK_LOG.clear()
    p, b = FakeServer(primary_behaviour), FakeServer(backup_behaviour)
    each = RunnableLambda(build_triage(p, b).invoke)
    out = each.batch([{"ticket": t} for t in tickets], config={"max_concurrency": 3}, return_exceptions=True)
    outcomes = {t: ("FAILED" if isinstance(r, Exception) else (r.category, r.urgency)) for t, r in zip(tickets, out)}
    return outcomes, p.hits, b.hits

print()
forward  = run_triage(TICKETS)
backward = run_triage(list(reversed(TICKETS)))

assert forward == backward, "results depend on ticket order!"
print(f"\nIdentical outcomes in both orders. Requests (primary, backup): {forward[1:]} forwards, {backward[1:]} backwards")


### The same pipeline against real providers

With real keys, only the two model lines change. Keep `max_retries=0` on whichever model you wrap with
`.with_retry()`, and keep the logging step:

```python
# primary = ChatGroq(model="llama-3.3-70b-versatile", max_retries=0, rate_limiter=limiter).with_structured_output(Ticket)
# backup  = ChatOpenAI(model="gpt-5-mini", max_retries=0).with_structured_output(Ticket)    # or your Gemini model
# triage  = triage_prompt | primary.with_retry(retry_if_exception_type=is_transient, stop_after_attempt=2) \
#                                  .with_fallbacks([note_fallback("backup model") | backup])
```

(That snippet was not run — it needs live keys. Note `is_transient` as written imports `groq` exception
classes; for a mixed Groq/OpenAI setup, the status-code branch is what does the work for API errors.)

### Try it yourself
1. In the capstone, set `stop_after_attempt=1`. How do the request counts change, and which tickets are
   affected?
2. Change `exceptions_to_handle` on the fallback to `(RateLimitError,)` only (import it from `groq`).
   What happens to the garbage ticket now — and is that better or worse?
3. Replace `note_fallback`'s `print` with a counter, and fail the run if more than 50% of tickets used the
   fallback. That is the beginning of an alert.
4. Add a `rate_limiter` to the primary model with `requests_per_second=2` and time the whole batch.
5. Make `FakeServer` return `503` for the first two requests and `200` after. Which retry settings recover?
6. Change `triage_each.batch(...)` back to `triage.batch(...)` and move the `POISON` ticket to the front of
   `TICKETS`. Compare the request counts with the table above — the answers will look the same.
7. Wrap `issue_refund_naive` in an `idempotency_key` that comes from the caller instead of a global set.
   Why is a caller-supplied key sturdier?

---
## You've finished the LangChain track

You can now call any model, shape its input and output, load and search your own documents, give it tools,
run it as an agent, and keep all of it standing when things break. Everything from here — stateful graphs,
multi-agent systems, long-term memory — is in your `LangGraph/` folders, and it builds directly on these
fourteen.
